In [14]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver

env_path = "/t9k/mnt/lxq/LangChain/dive-into-langgraph-main/.env"
_ = load_dotenv(dotenv_path=env_path)

# 加载模型
model = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    model="qwen3-coder-plus",
    temperature=0.7,
)

# 创建助手节点
def assistant(state: MessagesState):
    return {'messages': [model.invoke(state['messages'])]}

In [15]:
# 创建短期记忆
checkpointer = InMemorySaver()

# 创建图
builder = StateGraph(MessagesState)

# 添加节点
builder.add_node('assistant', assistant)

# 添加边
builder.add_edge(START, 'assistant')
builder.add_edge('assistant', END)

graph = builder.compile(checkpointer=checkpointer)

# 告诉智能体我叫 luochang
result = graph.invoke(
    {'messages': ['hi! i am luochang']},
    {"configurable": {"thread_id": "1"}},
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

hi! i am luochang
================================== Ai Message ==================================

Hello, Luochang! It's nice to meet you. Is there anything I can help you with today? Whether it's answering questions, providing information, or just having a chat - feel free to let me know what's on your mind! 😊


In [16]:
# 让智能体说出我的名字
result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    {"configurable": {"thread_id": "1"}},  
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

hi! i am luochang
================================== Ai Message ==================================

Hello, Luochang! It's nice to meet you. Is there anything I can help you with today? Whether it's answering questions, providing information, or just having a chat - feel free to let me know what's on your mind! 😊
================================ Human Message =================================

What is my name?
================================== Ai Message ==================================

Your name is Luochang, as you mentioned when you first greeted me! That's a nice name - though I should mention that you're the one who told me this, so you know your own name best. Is there something specific about your name you'd like to discuss or explore?


In [17]:
from langchain.agents import create_agent

# 创建短期记忆
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    checkpointer=checkpointer
)

# 告诉智能体我叫 luochang
result = agent.invoke(
    {'messages': ['hi! i am luochang']},
    {"configurable": {"thread_id": "2"}},
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

hi! i am luochang
================================== Ai Message ==================================

Hello, Luochang! It's nice to meet you. How can I assist you today?


In [18]:
# 让智能体说出我的名字
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    {"configurable": {"thread_id": "2"}},  
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

hi! i am luochang
================================== Ai Message ==================================

Hello, Luochang! It's nice to meet you. How can I assist you today?
================================ Human Message =================================

What is my name?
================================== Ai Message ==================================

Your name is Luochang, as you mentioned earlier. Is there anything else you would like to know or discuss?


In [19]:
# 删除SQLite数据库
if os.path.exists("short-memory.db"):
    os.remove("short-memory.db")

In [20]:
import os
import sqlite3

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents import create_agent

# 加载模型配置
_ = load_dotenv()

# 加载模型
model = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    model="qwen3-coder-plus",
    temperature=0.7,
)

# 创建sqlite支持的短期记忆
checkpointer = SqliteSaver(
    sqlite3.connect("short-memory.db", check_same_thread=False)
)

# 创建Agent
agent = create_agent(
    model=model,
    checkpointer=checkpointer,
)

# 告诉智能体我叫 luochang
result = agent.invoke(
    {'messages': ['hi! i am luochang']},
    {"configurable": {"thread_id": "3"}},
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

hi! i am luochang
================================== Ai Message ==================================

Hello, Luochang! It's nice to meet you. Is there anything I can help you with today? Whether it's answering questions, providing information, or just having a chat, I'm here for you! 😊


In [21]:
import os
import sqlite3

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents import create_agent

# 加载模型配置
_ = load_dotenv()

# 加载模型
model = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    model="qwen3-coder-plus",
    temperature=0.7,
)

# 创建sqlite支持的短期记忆
checkpointer = SqliteSaver(
    sqlite3.connect("short-memory.db", check_same_thread=False)
)

# 创建Agent
agent = create_agent(
    model=model,
    checkpointer=checkpointer,
)

# 让智能体回忆我的名字
result = agent.invoke(
    {'messages': ['What is my name?']},
    {"configurable": {"thread_id": "3"}},
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

hi! i am luochang
================================== Ai Message ==================================

Hello, Luochang! It's nice to meet you. Is there anything I can help you with today? Whether it's answering questions, providing information, or just having a chat, I'm here for you! 😊
================================ Human Message =================================

What is my name?
================================== Ai Message ==================================

Your name is Luochang, as you mentioned when you first greeted me. Hello again, Luochang! Is there anything specific you'd like to talk about or any questions you have?


长期记忆

In [22]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from langchain_core.runnables import RunnableConfig
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from dataclasses import dataclass

EMBED_MODEL = "text-embedding-v4"
EMBED_DIM = 1024

# 加载模型配置
_ = load_dotenv()

# 用于获取text embedding的接口
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

# 加载模型
model = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    model="qwen3-coder-plus",
    temperature=0.7,
)

In [23]:
# embedding生成函数
def embed(texts: list[str]) -> list[list[float]]:
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts,
        dimensions=EMBED_DIM,
    )

    return [item.embedding for item in response.data]

# 测试能否正常生成text embedding
texts = [
    "LangGraph的中间件非常强大",
    "LangGraph的MCP也很好用",
]
vectors = embed(texts)

len(vectors), len(vectors[0])

(2, 1024)

In [24]:
# InMemoryStore saves data to an in-memory dictionary. Use a DB-backed store in production use.
store = InMemoryStore(index={"embed": embed, "dims": EMBED_DIM})

# 添加两条用户数据
namespace = ("users", )
key = "user_1"
store.put(
    namespace,
    key,
    {
        "rules": [
            "User likes short, direct language",
            "User only speaks English & python",
        ],
        "rule_id": "3",
    },
)

store.put( 
    ("users",),  # Namespace to group related data together (users namespace for user data)
    "user_2",  # Key within the namespace (user ID as key)
    {
        "name": "John Smith",
        "language": "English",
    }  # Data to store for the given user
)

# get the "memory" by ID
item = store.get(namespace, "a-memory") 

# search for "memories" within this namespace, filtering on content equivalence, sorted by vector similarity
items = store.search( 
    namespace, filter={"rule_id": "3"}, query="language preferences"
)

items

[Item(namespace=['users'], key='user_1', value={'rules': ['User likes short, direct language', 'User only speaks English & python'], 'rule_id': '3'}, created_at='2026-03-06T04:15:31.003727+00:00', updated_at='2026-03-06T04:15:31.003736+00:00', score=0.40857101546618274)]

In [25]:
@dataclass
class Context:
    user_id: str

@tool
def get_user_info(runtime: ToolRuntime[Context]) -> str:
    """Look up user info."""
    # Access the store - same as that provided to `create_agent`
    store = runtime.store 
    user_id = runtime.context.user_id
    # Retrieve data from store - returns StoreValue object with value and metadata
    user_info = store.get(("users",), user_id) 
    return str(user_info.value) if user_info else "Unknown user"

agent = create_agent(
    model=model,
    tools=[get_user_info],
    # Pass store to agent - enables agent to access store when running tools
    store=store, 
    context_schema=Context
)

# Run the agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": "look up user information"}]},
    context=Context(user_id="user_2") 
)

for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

look up user information
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_dc06cc8df3ea4c84a57a794a)
 Call ID: call_dc06cc8df3ea4c84a57a794a
  Args:
================================= Tool Message =================================
Name: get_user_info

{'name': 'John Smith', 'language': 'English'}
================================== Ai Message ==================================

The user information found is as follows:

- **Name**: John Smith
- **Language**: English


In [26]:
from typing_extensions import TypedDict

# InMemoryStore saves data to an in-memory dictionary. Use a DB-backed store in production.
store = InMemoryStore() 

@dataclass
class Context:
    user_id: str

# TypedDict defines the structure of user information for the LLM
class UserInfo(TypedDict):
    name: str

# Tool that allows agent to update user information (useful for chat applications)
@tool
def save_user_info(user_info: UserInfo, runtime: ToolRuntime[Context]) -> str:
    """Save user info."""
    # Access the store - same as that provided to `create_agent`
    store = runtime.store 
    user_id = runtime.context.user_id 
    # Store data in the store (namespace, key, data)
    store.put(("users",), user_id, user_info) 
    return "Successfully saved user info."

agent = create_agent(
    model=model,
    tools=[save_user_info],
    store=store,
    context_schema=Context
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "My name is John Smith"}]},
    # user_id passed in context to identify whose information is being updated
    context=Context(user_id="user_123") 
)

# You can access the store directly to get the value
store.get(("users",), "user_123").value

{'name': 'John Smith'}